In [ ]:
# This notebook will be specifically for exported snapchat media
# The filenames are of the format: YYYY-MM-DD-<id> and suffixed with -main or -overlay for images/captions
# Note this may only work for images - will update this comment when I know
# Older snapchat memories do not contain EXIF data (tested so far: 2017, 2018)

# The plan for this notebook is to:
## - Iterate through the files in the specified folder (images and videos if possible), 
## - Identify the datetime that appears in the 'Created' field in explorer properties
## - [optional] Compare that it is the same date as in the filename (OTT?)
## - Format the datetime retrieved from the properties and store it as filename
## - Save to new location
## - Add this datetime to the EXIF data of the file
## - Remove old file / Move to a 'successfully copied' folder

In [ ]:
## Defining packages
import os
import shutil
import pathlib
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path
import piexif
import re


## Defining Static Variables for EXIF data
datetime_original_tag = 36867
datetime_digitized_tag = 36868
datetime_tag = 306

In [ ]:
source_base = "C:/Users/laram/Documents/Snapchat Exports Raw/20172018/2018-11-06/"
destination_base = "C:/Users/laram/Documents/Snapchat Exports Modified/20172018/2018-11-06/"
problematic_destination = "C:/Users/laram/Documents/Snapchat Exports Modified/20172018 - Problematic/"

In [ ]:
def has_exif_datetime_already(image_path):
    
    img = Image.open(image_path)
    exif_data = img._getexif()
    
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    img_datetime = exif_data.get(datetime_tag)

    true_datetime = original_datetime or digitized_datetime or img_datetime
    
    if true_datetime:
        return True
    else:
        return False

In [ ]:
### This cell prints the formatted (filename-suitable) timestamp for every file

# folder = Path(source_base)

# for file in folder.rglob("*"):
#     if file.is_file():
#         created_datetime = datetime.fromtimestamp(file.stat().st_ctime)
#         formatted_datetime = created_datetime.strftime("%Y-%m-%d_%H%M%S")

#         print(formatted_datetime)

In [ ]:
### This cell will copy and rename each file into it's new folder, and add EXIF data if it is a JPG/JPEG

total_files_processed = 0
file_names = []
new_file_datetimes = []
folder = Path(source_base)
for file in folder.rglob("*"):
    
    total_files_processed +=1
    
    file_counter = 0
    
    file_datetime = datetime.fromtimestamp(file.stat().st_ctime)
    file_datetime_string = file_datetime.strftime("%Y-%m-%d_%H%M%S")
    exif_suitable_datetime = file_datetime.strftime("%Y:%m:%d %H:%M:%S")
    new_file_datetimes.append(file_datetime_string)
    
    # ignoring any files that don't have the correct date
    if file_datetime.year == 2026:
        shutil.copy2(Path(str(file)), problematic_destination + Path(str(file)).name)
        continue
    
    ## Defining file name 
    
    file_name = str(file)
    file_names.append(file_name)
    
    original_file_name = Path(file_name).name
    original_file_stem = Path(file_name).stem
    original_file_suffix = Path(file_name).suffix
        
    if original_file_stem.endswith("-main"):
        new_file_name = f"{file_datetime_string}_{file_counter}_main{original_file_suffix}"

    elif original_file_stem.endswith("-overlay"):
        new_file_name = f"{file_datetime_string}_{file_counter}_overlay{original_file_suffix}"

    else:
        new_file_name = f"{file_datetime_string}_{file_counter}{original_file_suffix}"
            
    while new_file_name in file_names:
                  
        file_counter += 1
        
        if original_file_stem.endswith("-main"):
            new_file_name = f"{file_datetime_string}_{file_counter}_main{original_file_suffix}"

        elif original_file_stem.endswith("-overlay"):
            new_file_name = f"{file_datetime_string}_{file_counter}_overlay{original_file_suffix}"

        else:
            new_file_name = f"{file_datetime_string}_{file_counter}{original_file_suffix}"
          
    print(str(total_files_processed), new_file_name)
        
    file_names.append(new_file_name)
    out_path = destination_base + new_file_name
    
    source = Path(file_name)
    
    suffix = Path(str(out_path)).suffix
    
    # Flag images that already have exif data in a different folder
    if suffix.upper() in ['.JPG', '.JPEG'] and has_exif_datetime_already(file_name):
        shutil.copy2(source, problematic_destination + original_file_name)
        
    else:
        #source.rename(out_path)
        shutil.copy2(source, out_path)
        
        ## Adding EXIF data if it's an image
        suffix = Path(str(out_path)).suffix
        if suffix.upper() in ['.JPG', '.JPEG']:
            with Image.open(out_path) as new_img:
                new_exif_data = new_img._getexif()
                new_exif_dict = piexif.load(new_img.info.get("exif", b""))

                new_exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal]= exif_suitable_datetime

                try:
                    new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized]
                except:
                    new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized] = exif_suitable_datetime

                try:
                    new_exif_dict["0th"][piexif.ImageIFD.DateTime]
                except:
                    new_exif_dict["0th"][piexif.ImageIFD.DateTime] = exif_suitable_datetime

                exif_bytes = piexif.dump(new_exif_dict)

                new_img.save(out_path, exif=exif_bytes)
                new_img.close()